# 08 · 面向对象 OOP

用类别 class 把「数据」与「操作数据的行为」绑在一起，学会以对象思维组织程序。

## 学习目标
- 理解类别（class）与实例（instance）的关系，能用 class 定义新的数据类型。
- 掌握构造器（constructor）`__init__` 与 `self` 的角色，正确初始化与访问实例属性。
- 能撰写实例方法（instance method），并串接调用达成方法链（method chaining）。
- 理解继承（inheritance）与 `super().__init__()` 的运作，会做方法覆写（method overriding）。
- 能依情境判断该用字典（dict）当数据容器，还是自订 class。

## 从字典到类别：为什么需要 class

类别（class）是一张「数据蓝图」，把同一种事物的数据与行为绑在一起。

新手常先用字典（dict）装一笔数据，例如一个点的座标。dict 能装数据，但「对数据做的操作」（如算距离）只能写成散落各处的独立函数，数据与行为各走各的。

当这种数据会被重复使用、又需要附带行为时，就该把数据与行为封装（encapsulation）在一起，这正是 class 的动机。class 定义出类型，依蓝图造出的个别对象叫实例（instance）。

In [ ]:
# 概念：用 dict 装一笔数据 vs 用 class 把数据与行为绑在一起

# 造一笔假的点座标数据当示范用
point_dict = {'x': 3, 'y': 4}                 # dict 能装数据，但行为要另外写

# 注意：算距离得写成独立函数，数据与行为散落两处
def distance_from_origin(p):
    return (p['x'] ** 2 + p['y'] ** 2) ** 0.5

print('dict 写法：', distance_from_origin(point_dict))

# 改写成 class 雏形：数据（x, y）与行为（dist）绑在同一个蓝图
class Point:
    def __init__(self, x, y):                 # 创建对象当下设置座标
        self.x = x
        self.y = y
    def dist(self):                           # 行为就住在对象身上
        return (self.x ** 2 + self.y ** 2) ** 0.5

p = Point(3, 4)                               # 依蓝图造出一个实例
print('class 写法：', p.dist())

## 定义类别：__init__ 与 self

`__init__` 是构造器（constructor），在创建对象的当下自动运行，负责设置对象的初始状态。

`self` 代表「这个对象自己」，是每个方法的第一个参数（调用时不用自己传）。把值存到 `self.属性名`，就成了这个实例专属的实例属性（instance attribute）。

形状：
```
class 类别名:
    def __init__(self, 参数...):
        self.属性 = 参数
```
用 `类别名(参数...)` 创建实例（instantiation）。

In [ ]:
# 概念：用 __init__ 与 self 初始化实例属性
class Rectangle:
    def __init__(self, width, height):        # 创建对象时自动调用
        self.width = width                    # 把宽存成这个对象自己的属性
        self.height = height                  # 把高存成这个对象自己的属性

# 注意：调用时不用自己传 self，Python 会把对象本身塞进去
r1 = Rectangle(4, 3)                          # 造第一个矩形
r2 = Rectangle(10, 2)                         # 造第二个，各自独立

# 技巧：用点记号读回每个实例自己的属性
print('r1：', r1.width, 'x', r1.height)
print('r2：', r2.width, 'x', r2.height)

## 实例方法与属性访问

实例方法（instance method）是「属于对象的函数」，定义在 class 内、第一个参数为 `self`，因此能直接读写自己的属性。

它和一般函数的差别在于：方法挂在对象身上，透过点记号（dot notation）`对象.方法()` 调用，操作的就是该对象的数据。方法可以用 `return` 回传计算结果，也可以就地修改属性，让对象不只装数据、还会做事。

In [ ]:
# 概念：实例方法读写自己的属性（area 计算、scale 就地修改）
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
    def area(self):                           # 读属性算面积后回传
        return self.width * self.height
    def scale(self, factor):                  # 就地放大尺寸（不回传）
        self.width = self.width * factor
        self.height = self.height * factor

r = Rectangle(4, 3)
print('原始面积：', r.area())                 # 透过点记号调用方法

r.scale(2)                                    # 改变对象内部状态
print('放大后尺寸：', r.width, 'x', r.height)
print('放大后面积：', r.area())

## 方法链 method chaining

方法链（method chaining）是让方法回传 `self`，使多个操作能在一行内串接调用。

因为回传的是对象自己，下一个点记号就接着对同一个可变对象（mutable object）操作。这种一路 `.add().upper()` 接下去的写法称为流畅接口（fluent interface），好处是读起来像一连串动作；取舍是串太长时不易调试，需自行拿捏可读性。

形状：
```
def 方法(self, ...):
    ...修改 self...
    return self
```

In [ ]:
# 概念：方法回传 self 以支持方法链
class TextBuilder:
    def __init__(self):
        self.text = ''                        # 初始为空字符串
    def add(self, s):
        self.text = self.text + s
        return self                           # 回传自己，才能继续往下接
    def upper(self):
        self.text = self.text.upper()
        return self

# 技巧：一行串接多个操作，像在描述一连串动作
result = TextBuilder().add('hello ').add('world').upper().text
print(result)

## 继承 inheritance 与 super()

继承（inheritance）让子类别（subclass）沿用父类别（parent class）的属性与方法，只补上差异，避免重复代码。

语法是 `class 子类(父类):`。子类别若也有自己的 `__init__`，要用 `super().__init__(...)` 先把父类别该初始化的部分做好，再加上自己添加的属性，否则父类别设置的属性会缺漏。

形状：
```
class 子类(父类):
    def __init__(self, ...):
        super().__init__(...)
        self.新属性 = ...
```

In [ ]:
# 概念：子类别用 super().__init__() 初始化父类别部分，再补自己的属性
class Shape:
    def __init__(self, name):
        self.name = name                      # 父类别负责共通属性

class Circle(Shape):                          # Circle 继承自 Shape
    def __init__(self, radius):
        super().__init__('circle')            # 先让父类别设好 name
        self.radius = radius                  # 再补上子类别专属属性
    def area(self):
        return 3.14159 * self.radius ** 2

c = Circle(5)
# 注意：name 来自父类别，radius 来自子类别，两者都在同一个对象上
print('名称（继承自父类别）：', c.name)
print('面积：', c.area())

## 方法覆写与惯例方法

方法覆写（method overriding）是子类别定义一个与父类别同名的方法，用自己的版本取代父类别的行为。

当多个子类别都提供同名方法（共同接口 common interface），就能用同一个方法名统一调用、各自跑出不同结果，这就是多态（polymorphism）的直觉。实务上常约定一个固定方法名（例如机器学习框架的 `forward`）作为各子类别的共同入口。

In [ ]:
# 概念：多个子类别覆写同名方法 area，用统一接口调用（多态）
class Shape:
    def area(self):                           # 父类别给出共同接口
        return 0

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):                           # 覆写成正方形的算法
        return self.side ** 2

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):                           # 覆写成圆形的算法
        return 3.14159 * self.radius ** 2

# 技巧：不同子类别放同一清单，用同一个方法名调用即可
shapes = [Square(4), Circle(3), Square(2)]
for s in shapes:
    print(type(s).__name__, '面积：', s.area())

## dict 还是 class：设计取舍

字典（dict）与类别（class）都能当数据容器（data container），选哪个是设计取舍。

判断准则：

| 情况 | 建议 |
|---|---|
| 临时、松散、字段不固定的数据 | 用 dict |
| 需要附带行为（方法） | 用 class |
| 结构固定、会被重复使用 | 用 class |
| 只是一次性传递少量键值 | 用 dict |

为简单数据硬写一堆 class 是过度设计（over-engineering），反而伤可读性与可维护性；两端都要避免极端。

In [ ]:
# 概念：同一笔学生成绩，用 dict 与 class 各表达一次，比较差异

# 用 dict：快、轻，但行为（算平均）得另写函数
student_dict = {'name': 'Amy', 'scores': [80, 90, 70]}
avg_dict = sum(student_dict['scores']) / len(student_dict['scores'])
print('dict 版平均：', avg_dict)

# 用 class：结构固定、行为内置，会重复使用时更清楚
class Student:
    def __init__(self, name, scores):
        self.name = name
        self.scores = scores
    def average(self):                        # 行为就住在对象身上
        return sum(self.scores) / len(self.scores)

s = Student('Amy', [80, 90, 70])
print('class 版平均：', s.average())
# 注意：临时用一次选 dict，固定结构又有行为选 class

## 练习

以下三题由浅到深，皆为复合型但简短，数据请在题目内自己造（不引用任何外部文件）。

In [ ]:
# TODO 1 ·（简单）一栋建筑对象（集成：class 定义 + 实例方法 instance method）
#   情境：把一栋建筑的楼高 height 与楼层数 floors 包成一个对象，问它的平均单层楼高。
#   要求：
#     1. 定义 Building 类别，用 __init__ 接收 height（公尺）与 floors 两个参数，存成实例属性。
#     2. 写实例方法 floor_height()，回传 height 除以 floors（平均单层楼高）。
#     3. 用仿真数字创建一个实例（例如 height=30、floors=10），用点记号调用方法并印出结果。
#   验收：应该看到印出平均单层楼高 3.0（公尺）。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）街廓建筑统计（集成：class 定义 + 实例方法 + 属性访问 + 方法链 method chaining）
#   情境：用一个对象收集同一街廓（block）内多栋建筑的占地面积，可连续加入后一次算总和。
#   要求：
#     1. 在题目内用 list 或 numpy 自造一批仿真占地面积（例如 [120, 95, 200, 60]）。
#     2. 定义 Block 类别，__init__ 初始化空清单属性 areas 与一个 count 属性（初值 0）。
#     3. 写 add(area) 方法：把面积加进清单、count 加一，并回传 self 以支持方法链。
#     4. 一行串接多次 add(...)，再调用 total() 回传占地面积总和，并用属性访问读出 count。
#   验收：应该看到占地面积总和（例如 475）与已加入栋数 count（例如 4）。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）容积率达标筛选（集成：继承 inheritance + super().__init__() + 方法覆写 + 共同接口 + 设计取舍）
#   情境：不同用途分类（住宅、商业）有不同的容积率 FAR 上限；政策情境下要找出哪些建筑超标。
#   要求：
#     1. 在题目内用 list 造几栋仿真建筑数据（楼地板面积 GFA、占地面积 footprint、用途标签）。
#     2. 定义父类别 Building，__init__ 存 GFA 与 footprint，提供共同方法名 far()（回传 GFA / footprint）与 over_limit()。
#     3. 衍生 Residential 与 Commercial 子类别，各用 super().__init__() 初始化，覆写 over_limit()：
#        以各自 FAR 上限（例如住宅 2.0、商业 5.0）判断是否超标。
#     4. 把多种子类别对象放进同一清单，循环调用同一个 over_limit()（多态），聚合出超标建筑清单。
#   验收：应该看到一份超标建筑清单，且不同用途各自套用自己的 FAR 上限做判断。
# TODO: 在这里完成

## 小结

- 类别（class）是数据蓝图，把数据与行为封装在一起；依蓝图造出的个别对象是实例（instance）。
- `__init__` 在创建对象时自动运行，`self` 代表对象自己，用 `self.属性` 设置实例属性。
- 实例方法挂在对象身上，透过点记号读写自己的属性；让方法回传 `self` 即可串成方法链。
- 继承（inheritance）让子类别沿用父类别并补上差异，子类别的 `__init__` 要先调用 `super().__init__()`。
- 方法覆写（method overriding）配合共同接口带来多态（polymorphism）：同一个方法名、各子类别各自的行为。
- 松散临时数据用 dict，固定结构又需要行为时用 class，避免过度设计。